# 02 - Feature Engineering and Splits

## Goal
Build the modeling table from monthly macro data, create a clean feature registry, and generate leakage-safe train/val/test splits with sequence-ready views.

This notebook uses the full project feature family and split conventions, with explicit saved artifacts for reproducibility.

In [1]:
from pathlib import Path
import sys
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.project_utils import ensure_project_dirs, seed_everything

seed_everything(42)
paths = ensure_project_dirs()

print("Repo root:", ROOT)
print("Processed dir:", paths["processed_data"])

Repo root: C:\Users\jijos\Desktop\261-Project
Processed dir: C:\Users\jijos\Desktop\261-Project\data\processed


## Config

- `FEATURE_SET` switches between a compact set and full set.
- `N_LAGS` is the main sequence lookback.
- `INFL_SMOOTHING` controls whether to use `Infl_ema`, `Infl_ma3/Infl_ma6`, or both.

In [2]:
FEATURE_SET = "full"         # options: "small", "full"
N_LAGS = 60                   # default lookback
N_LAGS_BACKTEST = 36          # backtest mode default
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
INFL_SMOOTHING = "ema"       # options: "ema", "ma", "both"
EMA_ALPHA = 0.30

# Split strategy
# - ratio: pure 70/15/15 chronological split
# - covid_aware: fixed trailing windows for val/test so train includes COVID-era months
SPLIT_STRATEGY = "covid_aware"   # options: "ratio", "covid_aware"
COVID_AWARE_VAL_MONTHS = 24
COVID_AWARE_TEST_MONTHS = 24
MIN_TRAIN_END_DATE = "2021-06-01"  # guardrail for covid_aware strategy

if FEATURE_SET not in {"small", "full"}:
    raise ValueError("FEATURE_SET must be 'small' or 'full'")
if INFL_SMOOTHING not in {"ema", "ma", "both"}:
    raise ValueError("INFL_SMOOTHING must be 'ema', 'ma', or 'both'")
if not np.isclose(TRAIN_RATIO + VAL_RATIO + TEST_RATIO, 1.0):
    raise ValueError("Split ratios must sum to 1.0")
if SPLIT_STRATEGY not in {"ratio", "covid_aware"}:
    raise ValueError("SPLIT_STRATEGY must be 'ratio' or 'covid_aware'")

print("FEATURE_SET:", FEATURE_SET)
print("N_LAGS:", N_LAGS)
print("N_LAGS_BACKTEST:", N_LAGS_BACKTEST)
print("INFL_SMOOTHING:", INFL_SMOOTHING)
print("SPLIT_STRATEGY:", SPLIT_STRATEGY)


FEATURE_SET: full
N_LAGS: 60
N_LAGS_BACKTEST: 36
INFL_SMOOTHING: ema
SPLIT_STRATEGY: covid_aware


## Load aligned dataset and target metadata

Primary input is `data/processed/aligned_macro_target.csv` from notebook `01`.

If missing, we fallback to `data/raw/fred_macro_monthly_raw.csv` and rebuild CPI inflation columns with a default target (`yoy`, horizon 1) so this notebook can still run.

In [3]:
processed_aligned_path = paths["processed_data"] / "aligned_macro_target.csv"
raw_cache_path = paths["raw_data"] / "fred_macro_monthly_raw.csv"
meta_path = paths["processed_data"] / "target_definition_metadata.json"

if processed_aligned_path.exists():
    full_df = pd.read_csv(processed_aligned_path, index_col=0, parse_dates=True)
    print(f"Loaded aligned processed data: {processed_aligned_path}")
else:
    if not raw_cache_path.exists():
        raise FileNotFoundError(
            "Missing aligned and raw data files. Run notebook 01 first or add cached files."
        )
    print("Aligned file missing. Building fallback from raw cache.")
    raw_df_tmp = pd.read_csv(raw_cache_path, index_col=0, parse_dates=True)
    raw_df_tmp = raw_df_tmp.sort_index()
    raw_df_tmp["CPI_inflation_mom"] = raw_df_tmp["CPI"].pct_change(1) * 100.0
    raw_df_tmp["CPI_inflation_yoy"] = raw_df_tmp["CPI"].pct_change(12) * 100.0
    raw_df_tmp["target_yoy_t_plus_1"] = raw_df_tmp["CPI_inflation_yoy"].shift(-1)
    full_df = raw_df_tmp.dropna(subset=["CPI_inflation_yoy", "target_yoy_t_plus_1"]).copy()
    full_df.to_csv(processed_aligned_path)
    print(f"Saved fallback aligned data: {processed_aligned_path}")

full_df = full_df.sort_index()
full_df.index = pd.to_datetime(full_df.index)
if full_df.index.has_duplicates:
    raise ValueError("Duplicate timestamps found in aligned dataframe.")

if meta_path.exists():
    target_meta = json.loads(meta_path.read_text(encoding="utf-8"))
else:
    target_meta = {
        "target_mode": "yoy",
        "target_name": "target_yoy_t_plus_1",
        "forecast_horizon_months": 1,
    }

TARGET_NAME = target_meta.get("target_name", "target_yoy_t_plus_1")
if TARGET_NAME not in full_df.columns:
    raise KeyError(f"Target column '{TARGET_NAME}' is missing from aligned dataframe.")

BASE_MACRO_COLS = [
    "CPI", "Unemployment", "InterestRate", "M2", "Treasury10Y", "OilPrice",
    "PPI", "RetailSales", "Sentiment", "Employment", "HousingStarts"
]

raw_df = full_df[BASE_MACRO_COLS].copy()
# keep current inflation series separate so feature creation is explicit
inflation_current = full_df[["CPI_inflation_mom", "CPI_inflation_yoy"]].copy()
target_series = full_df[TARGET_NAME].copy()

print("raw_df shape:", raw_df.shape)
print("target name:", TARGET_NAME)
print("date range:", raw_df.index.min().date(), "to", raw_df.index.max().date())

Loaded aligned processed data: C:\Users\jijos\Desktop\261-Project\data\processed\aligned_macro_target.csv


raw_df shape: (483, 11)
target name: target_yoy_t_plus_1
date range: 1986-01-01 to 2026-03-01


## Feature families and why they exist

- **Policy/labor deltas** (`Unemp_d`, `Rate_d`, `Employment_d`, `Housing_d`): capture short-run macro momentum.
- **Price and liquidity pressure** (`Oil_ret`, `PPI_yoy`, `M2_yoy`, `Retail_yoy`): map input costs, money growth, and demand.
- **Levels and sentiment** (`T10Y`, `Sentiment`): capture market rate regime and consumer expectations.
- **Nonlinear terms** (`*_sq`, interactions): allow asymmetric inflation response under shocks.
- **Inflation dynamics** (`Infl_ema` or `Infl_ma3/Infl_ma6`, `Infl_vol6`, `Inflation_prev`): persistence and short-run volatility.
- **Seasonality and shock dummies** (`MoY_sin`, `MoY_cos`, `COVID`): cyclical pattern and COVID-era regime break.

In [4]:
def engineer_features(raw_df: pd.DataFrame, inflation_current: pd.DataFrame, smoothing: str = "ema", ema_alpha: float = 0.30) -> pd.DataFrame:
    df = pd.DataFrame(index=raw_df.index)

    # core transforms
    df["Unemp_d"] = raw_df["Unemployment"].diff()
    df["Rate_d"] = raw_df["InterestRate"].diff()
    df["Oil_ret"] = np.log(raw_df["OilPrice"]).diff() * 100.0
    df["PPI_yoy"] = raw_df["PPI"].pct_change(12) * 100.0
    df["M2_yoy"] = raw_df["M2"].pct_change(12) * 100.0
    df["Retail_yoy"] = raw_df["RetailSales"].pct_change(12) * 100.0
    df["Employment_d"] = raw_df["Employment"].diff()
    df["Housing_d"] = raw_df["HousingStarts"].diff()
    df["T10Y"] = raw_df["Treasury10Y"]
    df["Sentiment"] = raw_df["Sentiment"]

    # nonlinear terms
    df["PPI_yoy_sq"] = df["PPI_yoy"] ** 2
    df["Oil_ret_sq"] = df["Oil_ret"] ** 2
    df["Rate_d_sq"] = df["Rate_d"] ** 2
    df["Unemp_d_sq"] = df["Unemp_d"] ** 2
    df["Rate_Unemp"] = df["Rate_d"] * df["Unemp_d"]
    df["Oil_PPI"] = df["Oil_ret"] * df["PPI_yoy"]

    # inflation dynamic terms from current-period inflation
    infl = inflation_current["CPI_inflation_yoy"] if TARGET_NAME.startswith("target_yoy") else inflation_current["CPI_inflation_mom"]

    if smoothing in {"ema", "both"}:
        df["Infl_ema"] = infl.ewm(alpha=ema_alpha, adjust=False).mean()
    if smoothing in {"ma", "both"}:
        df["Infl_ma3"] = infl.rolling(3).mean()
        df["Infl_ma6"] = infl.rolling(6).mean()

    df["Infl_vol6"] = infl.rolling(6).std()
    df["Inflation_prev"] = infl

    # seasonality
    month = df.index.month
    df["MoY_sin"] = np.sin(2 * np.pi * month / 12.0)
    df["MoY_cos"] = np.cos(2 * np.pi * month / 12.0)

    # covid dummy
    covid_start = pd.Timestamp("2020-03-01")
    covid_end = pd.Timestamp("2021-06-30")
    df["COVID"] = ((df.index >= covid_start) & (df.index <= covid_end)).astype(int)

    return df


engineered_df = engineer_features(raw_df, inflation_current, smoothing=INFL_SMOOTHING, ema_alpha=EMA_ALPHA)
print("engineered_df shape before alignment drop:", engineered_df.shape)
engineered_df.head(3)

engineered_df shape before alignment drop: (483, 22)


C:\Users\jijos\AppData\Local\Temp\ipykernel_14516\236851052.py:8: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["PPI_yoy"] = raw_df["PPI"].pct_change(12) * 100.0
C:\Users\jijos\AppData\Local\Temp\ipykernel_14516\236851052.py:9: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["M2_yoy"] = raw_df["M2"].pct_change(12) * 100.0
C:\Users\jijos\AppData\Local\Temp\ipykernel_14516\236851052.py:10: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_

,Unemp_d,Rate_d,Oil_ret,PPI_yoy,M2_yoy,Retail_yoy,Employment_d,Housing_d,T10Y,Sentiment,...,Rate_d_sq,Unemp_d_sq,Rate_Unemp,Oil_PPI,Infl_ema,Infl_vol6,Inflation_prev,MoY_sin,MoY_cos,COVID
DATE,,,,,,,,,,,,,,,,,,,,,
1986-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.19,95.6,...,NaN,NaN,NaN,NaN,3.886256,NaN,3.886256,0.500000,8.660254e-01,0
1986-02-01,0.5,-0.28,-39.433230,NaN,NaN,NaN,115.0,-124.0,8.70,95.9,...,0.0784,0.25,-0.14,NaN,3.654341,NaN,3.113208,0.866025,5.000000e-01,0
1986-03-01,0.0,-0.38,-20.322716,NaN,NaN,NaN,87.0,28.0,7.78,95.1,...,0.1444,0.00,-0.00,NaN,3.234731,NaN,2.255639,1.000000,6.123234e-17,0


## Feature registry

- `full`: full engineered feature family.
- `small`: compact subset for lighter experiments.

Feature names and descriptions are saved to disk for reproducibility.

In [5]:
feature_descriptions = {
    "Unemp_d": "Monthly change in unemployment rate",
    "Rate_d": "Monthly change in policy rate",
    "Oil_ret": "Monthly log return of oil price",
    "PPI_yoy": "Year-over-year producer price inflation",
    "M2_yoy": "Year-over-year money supply growth",
    "Retail_yoy": "Year-over-year retail sales growth",
    "Employment_d": "Monthly change in payroll employment",
    "Housing_d": "Monthly change in housing starts",
    "T10Y": "10-year treasury yield level",
    "Sentiment": "Consumer sentiment index level",
    "PPI_yoy_sq": "Squared PPI inflation",
    "Oil_ret_sq": "Squared oil return",
    "Rate_d_sq": "Squared rate change",
    "Unemp_d_sq": "Squared unemployment change",
    "Rate_Unemp": "Interaction of rate and unemployment changes",
    "Oil_PPI": "Interaction of oil return and PPI inflation",
    "Infl_ema": "Exponential moving average of current inflation",
    "Infl_ma3": "3-month moving average of current inflation",
    "Infl_ma6": "6-month moving average of current inflation",
    "Infl_vol6": "6-month rolling inflation volatility",
    "Inflation_prev": "Current inflation value (acts as AR term)",
    "MoY_sin": "Month-of-year seasonal sine component",
    "MoY_cos": "Month-of-year seasonal cosine component",
    "COVID": "COVID regime dummy (2020-03 to 2021-06)",
}

full_features = [
    "Unemp_d", "Rate_d", "Oil_ret", "PPI_yoy", "M2_yoy", "Retail_yoy", "Employment_d",
    "Housing_d", "T10Y", "Sentiment", "PPI_yoy_sq", "Oil_ret_sq", "Rate_d_sq", "Unemp_d_sq",
    "Rate_Unemp", "Oil_PPI", "Infl_vol6", "Inflation_prev", "MoY_sin", "MoY_cos", "COVID"
]

if INFL_SMOOTHING == "ema":
    full_features = full_features + ["Infl_ema"]
elif INFL_SMOOTHING == "ma":
    full_features = full_features + ["Infl_ma3", "Infl_ma6"]
else:
    full_features = full_features + ["Infl_ema", "Infl_ma3", "Infl_ma6"]

small_features = [
    "Unemp_d", "Oil_ret", "PPI_yoy", "Employment_d", "Housing_d", "T10Y", "Sentiment",
    "Inflation_prev", "Infl_vol6", "MoY_sin", "MoY_cos", "COVID"
]
if "Infl_ema" in full_features:
    small_features.append("Infl_ema")

feature_registry = {
    "small": [f for f in small_features if f in engineered_df.columns],
    "full": [f for f in full_features if f in engineered_df.columns],
}

selected_features = feature_registry[FEATURE_SET]

if len(selected_features) == 0:
    raise ValueError("Selected feature set is empty.")

registry_json = paths["processed_data"] / "feature_registry.json"
registry_table_csv = paths["processed_data"] / "feature_registry_table.csv"

registry_payload = {
    "selected_set": FEATURE_SET,
    "selected_features": selected_features,
    "feature_registry": feature_registry,
    "feature_descriptions": feature_descriptions,
}
registry_json.write_text(json.dumps(registry_payload, indent=2), encoding="utf-8")

registry_table = pd.DataFrame([
    {"feature": f, "description": feature_descriptions.get(f, "")}
    for f in selected_features
])
registry_table.to_csv(registry_table_csv, index=False)

print("Selected feature count:", len(selected_features))
print("Selected features:", selected_features)
print("Saved:", registry_json)
print("Saved:", registry_table_csv)
registry_table.head(10)

Selected feature count: 22
Selected features: ['Unemp_d', 'Rate_d', 'Oil_ret', 'PPI_yoy', 'M2_yoy', 'Retail_yoy', 'Employment_d', 'Housing_d', 'T10Y', 'Sentiment', 'PPI_yoy_sq', 'Oil_ret_sq', 'Rate_d_sq', 'Unemp_d_sq', 'Rate_Unemp', 'Oil_PPI', 'Infl_vol6', 'Inflation_prev', 'MoY_sin', 'MoY_cos', 'COVID', 'Infl_ema']
Saved: C:\Users\jijos\Desktop\261-Project\data\processed\feature_registry.json
Saved: C:\Users\jijos\Desktop\261-Project\data\processed\feature_registry_table.csv


,feature,description
0,Unemp_d,Monthly change in unemployment rate
1,Rate_d,Monthly change in policy rate
2,Oil_ret,Monthly log return of oil price
3,PPI_yoy,Year-over-year producer price inflation
4,M2_yoy,Year-over-year money supply growth
5,Retail_yoy,Year-over-year retail sales growth
6,Employment_d,Monthly change in payroll employment
7,Housing_d,Monthly change in housing starts
8,T10Y,10-year treasury yield level
9,Sentiment,Consumer sentiment index level


## Leakage-safe modeling table

Rules used:
- features are built from data at time `t` or earlier,
- target is already defined in notebook `01` as future inflation at `t+1`,
- rows with incomplete rolling features are dropped once at the end.

This avoids look-ahead leakage.

In [6]:
model_df = pd.concat([engineered_df[selected_features], target_series.rename(TARGET_NAME)], axis=1)
model_df = model_df.dropna().copy()

# Final NaN safety check (required)
assert model_df.isna().sum().sum() == 0, "NaNs remain in final modeling table"

feature_table_csv = paths["processed_data"] / f"modeling_table_{FEATURE_SET}.csv"
model_df.to_csv(feature_table_csv, index=True)

feature_table_parquet = paths["processed_data"] / f"modeling_table_{FEATURE_SET}.parquet"
parquet_saved = False
try:
    model_df.to_parquet(feature_table_parquet, index=True)
    parquet_saved = True
except Exception as e:
    print("Parquet save skipped:", repr(e))

print("model_df shape:", model_df.shape)
print("Saved CSV:", feature_table_csv)
print("Saved Parquet:", feature_table_parquet if parquet_saved else "not saved")
model_df.head(3)

model_df shape: (395, 23)
Saved CSV: C:\Users\jijos\Desktop\261-Project\data\processed\modeling_table_full.csv
Saved Parquet: C:\Users\jijos\Desktop\261-Project\data\processed\modeling_table_full.parquet


,Unemp_d,Rate_d,Oil_ret,PPI_yoy,M2_yoy,Retail_yoy,Employment_d,Housing_d,T10Y,Sentiment,...,Unemp_d_sq,Rate_Unemp,Oil_PPI,Infl_vol6,Inflation_prev,MoY_sin,MoY_cos,COVID,Infl_ema,target_yoy_t_plus_1
DATE,,,,,,,,,,,,,,,,,,,,,
1993-01-01,-0.1,0.10,-1.987736,2.076125,1.120904,6.958341,288.0,-17.0,6.60,89.3,...,0.01,-0.010,-4.126789,0.135881,3.258508,0.500000,8.660254e-01,0,3.104348,3.246753
1993-02-01,-0.2,0.01,5.390681,2.068966,0.426471,6.422881,257.0,0.0,6.26,86.6,...,0.04,-0.002,11.153134,0.149483,3.246753,0.866025,5.000000e-01,0,3.147070,3.086863
1993-03-01,-0.1,0.04,1.168401,2.239449,0.229149,5.914016,-43.0,-127.0,5.98,85.9,...,0.01,-0.004,2.616575,0.138817,3.086863,1.000000,6.123234e-17,0,3.129008,3.225806


## Chronological 70/15/15 splits + context windows

Split logic:
- train = first 70%
- val = next 15%
- test = final 15%

For sequence models, val/test blocks include a left context of `N_LAGS` rows so their first window has full history.

In [7]:
def compute_split_bounds(
    n_rows: int,
    index: pd.Index,
    strategy: str = "ratio",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    val_months: int = 24,
    test_months: int = 24,
    min_train_end_date: str | None = None,
) -> tuple[int, int]:
    if strategy == "ratio":
        train_end = int(n_rows * train_ratio)
        val_end = int(n_rows * (train_ratio + val_ratio))
        return train_end, val_end

    # covid-aware strategy: keep recent years for val/test, leave earlier history (including COVID) for train
    if strategy == "covid_aware":
        if val_months <= 0 or test_months <= 0:
            raise ValueError("val_months and test_months must be positive for covid_aware strategy")
        if val_months + test_months >= n_rows:
            raise ValueError("val_months + test_months must be smaller than dataset length")

        val_end = n_rows - test_months
        train_end = val_end - val_months

        if min_train_end_date is not None:
            min_dt = pd.Timestamp(min_train_end_date)
            if index[train_end - 1] < min_dt:
                # shift boundary right to satisfy minimum date while preserving test size
                shifted_train_end = int(index.searchsorted(min_dt, side="right"))
                max_train_end = val_end - 1
                train_end = min(max(shifted_train_end, 1), max_train_end)

        return train_end, val_end

    raise ValueError(f"Unknown strategy: {strategy}")


def make_sequences(X_2d: np.ndarray, y_1d: np.ndarray, n_lags: int) -> tuple[np.ndarray, np.ndarray]:
    X_out, y_out = [], []
    for i in range(n_lags, len(X_2d)):
        X_out.append(X_2d[i - n_lags : i, :])
        y_out.append(y_1d[i])
    return np.asarray(X_out), np.asarray(y_out)


def flatten_sequences_for_tabular(X_3d: np.ndarray) -> np.ndarray:
    if X_3d.ndim != 3:
        raise ValueError("Expected 3D sequence array")
    n, lags, feats = X_3d.shape
    return X_3d.reshape(n, lags * feats)


X_df = model_df[selected_features]
y_sr = model_df[TARGET_NAME]

n_total = len(model_df)
train_end, val_end = compute_split_bounds(
    n_rows=n_total,
    index=X_df.index,
    strategy=SPLIT_STRATEGY,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    val_months=COVID_AWARE_VAL_MONTHS,
    test_months=COVID_AWARE_TEST_MONTHS,
    min_train_end_date=MIN_TRAIN_END_DATE,
)

# chronological split assertions
assert 0 < train_end < val_end < n_total, "Invalid split boundaries"
assert X_df.index.is_monotonic_increasing, "Index is not chronological"

# base splits
X_train_df = X_df.iloc[:train_end]
X_val_df = X_df.iloc[train_end:val_end]
X_test_df = X_df.iloc[val_end:]

y_train_sr = y_sr.iloc[:train_end]
y_val_sr = y_sr.iloc[train_end:val_end]
y_test_sr = y_sr.iloc[val_end:]

# context starts for sequence blocks
val_ctx_start = max(0, train_end - N_LAGS)
test_ctx_start = max(0, val_end - N_LAGS)

# sequence arrays
X_train_seq, y_train_seq = make_sequences(X_train_df.values, y_train_sr.values, N_LAGS)

X_val_block = X_df.iloc[val_ctx_start:val_end].values
y_val_block = y_sr.iloc[val_ctx_start:val_end].values
X_val_all, y_val_all = make_sequences(X_val_block, y_val_block, N_LAGS)
keep_val_from = max(0, train_end - (val_ctx_start + N_LAGS))
X_val_seq = X_val_all[keep_val_from:]
y_val_seq = y_val_all[keep_val_from:]

X_test_block = X_df.iloc[test_ctx_start:].values
y_test_block = y_sr.iloc[test_ctx_start:].values
X_test_all, y_test_all = make_sequences(X_test_block, y_test_block, N_LAGS)
keep_test_from = max(0, val_end - (test_ctx_start + N_LAGS))
X_test_seq = X_test_all[keep_test_from:]
y_test_seq = y_test_all[keep_test_from:]

# required sequence count assertion
assert len(X_train_seq) > 0 and len(X_val_seq) > 0 and len(X_test_seq) > 0, "One or more sequence splits are empty"

# reusable tabular view for tree models
X_train_tab = flatten_sequences_for_tabular(X_train_seq)
X_val_tab = flatten_sequences_for_tabular(X_val_seq)
X_test_tab = flatten_sequences_for_tabular(X_test_seq)

print("Split strategy:", SPLIT_STRATEGY)
print("Split row counts:", {"train": len(X_train_df), "val": len(X_val_df), "test": len(X_test_df)})
print("Split date ranges:", {
    "train": (str(X_train_df.index.min().date()), str(X_train_df.index.max().date())),
    "val": (str(X_val_df.index.min().date()), str(X_val_df.index.max().date())),
    "test": (str(X_test_df.index.min().date()), str(X_test_df.index.max().date())),
})
print("Sequence counts:", {"train": len(X_train_seq), "val": len(X_val_seq), "test": len(X_test_seq)})
print("Tabular flattened shape (train):", X_train_tab.shape)


Split strategy: covid_aware
Split row counts: {'train': 347, 'val': 24, 'test': 24}
Split date ranges: {'train': ('1993-01-01', '2021-11-01'), 'val': ('2021-12-01', '2023-11-01'), 'test': ('2023-12-01', '2026-01-01')}
Sequence counts: {'train': 287, 'val': 24, 'test': 24}
Tabular flattened shape (train): (287, 1320)


In [8]:
summary_rows = [
    {
        "split": "train",
        "row_count": len(X_train_df),
        "seq_count": len(X_train_seq),
        "seq_shape": str(X_train_seq.shape),
        "tabular_shape": str(X_train_tab.shape),
        "start_date": str(X_train_df.index.min().date()),
        "end_date": str(X_train_df.index.max().date()),
    },
    {
        "split": "val",
        "row_count": len(X_val_df),
        "seq_count": len(X_val_seq),
        "seq_shape": str(X_val_seq.shape),
        "tabular_shape": str(X_val_tab.shape),
        "start_date": str(X_val_df.index.min().date()),
        "end_date": str(X_val_df.index.max().date()),
    },
    {
        "split": "test",
        "row_count": len(X_test_df),
        "seq_count": len(X_test_seq),
        "seq_shape": str(X_test_seq.shape),
        "tabular_shape": str(X_test_tab.shape),
        "start_date": str(X_test_df.index.min().date()),
        "end_date": str(X_test_df.index.max().date()),
    },
]

sequence_summary_df = pd.DataFrame(summary_rows)
sequence_summary_path = paths["processed_data"] / "sequence_shape_summary.csv"
sequence_summary_df.to_csv(sequence_summary_path, index=False)

split_meta = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "target_name": TARGET_NAME,
    "feature_set": FEATURE_SET,
    "n_lags": N_LAGS,
    "n_lags_backtest_default": N_LAGS_BACKTEST,
    "split_strategy": SPLIT_STRATEGY,
    "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
    "covid_aware_windows": {"val_months": COVID_AWARE_VAL_MONTHS, "test_months": COVID_AWARE_TEST_MONTHS},
    "min_train_end_date": MIN_TRAIN_END_DATE,
    "boundaries": {
        "train_end_idx": int(train_end),
        "val_end_idx": int(val_end),
        "train_end_date": str(X_df.index[train_end - 1].date()),
        "val_end_date": str(X_df.index[val_end - 1].date()),
    },
    "counts": {
        "rows": {"train": int(len(X_train_df)), "val": int(len(X_val_df)), "test": int(len(X_test_df))},
        "sequences": {"train": int(len(X_train_seq)), "val": int(len(X_val_seq)), "test": int(len(X_test_seq))},
    },
    "context": {
        "val_ctx_start_idx": int(val_ctx_start),
        "test_ctx_start_idx": int(test_ctx_start),
    },
    "selected_features": selected_features,
}

split_meta_path = paths["processed_data"] / "split_metadata.json"
split_meta_path.write_text(json.dumps(split_meta, indent=2), encoding="utf-8")

print("Saved:", split_meta_path)
print("Saved:", sequence_summary_path)
sequence_summary_df

Saved: C:\Users\jijos\Desktop\261-Project\data\processed\split_metadata.json
Saved: C:\Users\jijos\Desktop\261-Project\data\processed\sequence_shape_summary.csv


,split,row_count,seq_count,seq_shape,tabular_shape,start_date,end_date
0,train,347,287,"(287, 60, 22)","(287, 1320)",1993-01-01,2021-11-01
1,val,24,24,"(24, 60, 22)","(24, 1320)",2021-12-01,2023-11-01
2,test,24,24,"(24, 60, 22)","(24, 1320)",2023-12-01,2026-01-01


## Verification checks

Required checks in this notebook:
- no NaNs in final modeling table,
- chronological split boundaries,
- non-zero sequence counts.

In [9]:
# already asserted above, repeated as explicit final checks
assert model_df.isna().sum().sum() == 0
assert X_train_df.index.max() < X_val_df.index.min()
assert X_val_df.index.max() < X_test_df.index.min()
assert len(X_train_seq) > 0 and len(X_val_seq) > 0 and len(X_test_seq) > 0

print("Final modeling table shape:", model_df.shape)
print("Selected feature set:", FEATURE_SET)
print("Selected feature count:", len(selected_features))
print("Target:", TARGET_NAME)
print("Chronological split check: PASS")
print("Sequence count check: PASS")

Final modeling table shape: (395, 23)
Selected feature set: full
Selected feature count: 22
Target: target_yoy_t_plus_1
Chronological split check: PASS
Sequence count check: PASS
